In [48]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.cluster import DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics.cluster import adjusted_rand_score

In [2]:
df = pd.read_csv('penguins.csv')

In [5]:
df.head()

,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
0,39.1,18.7,181.0,3750.0,MALE
1,39.5,17.4,186.0,3800.0,FEMALE
2,40.3,18.0,195.0,3250.0,FEMALE
3,NaN,NaN,NaN,NaN,NaN
4,36.7,19.3,193.0,3450.0,FEMALE


In [4]:
df.describe()

,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g
count,342.000000,342.000000,342.000000,342.000000
mean,43.921930,17.151170,214.014620,4201.754386
std,5.459584,1.974793,260.558057,801.954536
min,32.100000,13.100000,-132.000000,2700.000000
25%,39.225000,15.600000,190.000000,3550.000000
50%,44.450000,17.300000,197.000000,4050.000000
75%,48.500000,18.700000,213.750000,4750.000000
max,59.600000,21.500000,5000.000000,6300.000000


In [6]:
df.describe(include='object')

,sex
count,335
unique,3
top,MALE
freq,169


In [8]:
df['sex'].unique()

array(['MALE', 'FEMALE', nan, '.'], dtype=object)

In [10]:
df = df[df['flipper_length_mm'] != -132]
df = df[df['flipper_length_mm'] != 5000]

In [11]:
df = df.drop(df[df['sex'] == '.'].index)

In [12]:
df.head()

,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
0,39.1,18.7,181.0,3750.0,MALE
1,39.5,17.4,186.0,3800.0,FEMALE
2,40.3,18.0,195.0,3250.0,FEMALE
3,NaN,NaN,NaN,NaN,NaN
4,36.7,19.3,193.0,3450.0,FEMALE


In [13]:
df.dropna(inplace=True)

In [14]:
df.head()

,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
0,39.1,18.7,181.0,3750.0,MALE
1,39.5,17.4,186.0,3800.0,FEMALE
2,40.3,18.0,195.0,3250.0,FEMALE
4,36.7,19.3,193.0,3450.0,FEMALE
5,39.3,20.6,190.0,3650.0,MALE


In [16]:
le = LabelEncoder()

In [17]:
df['sex'] = le.fit_transform(df['sex'])

In [18]:
df.head()

,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
0,39.1,18.7,181.0,3750.0,1
1,39.5,17.4,186.0,3800.0,0
2,40.3,18.0,195.0,3250.0,0
4,36.7,19.3,193.0,3450.0,0
5,39.3,20.6,190.0,3650.0,1


In [19]:
from sklearn.cluster import KMeans

In [20]:
kmeans = KMeans(n_clusters=2, random_state=0).fit(df)

In [23]:
# kmeans.labels_

In [22]:
df['cluster_label'] = kmeans.labels_
display(df.head())

,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex,cluster_label
0,39.1,18.7,181.0,3750.0,1,1
1,39.5,17.4,186.0,3800.0,0,1
2,40.3,18.0,195.0,3250.0,0,1
4,36.7,19.3,193.0,3450.0,0,1
5,39.3,20.6,190.0,3650.0,1,1


In [25]:
import plotly.express as px
from sklearn.cluster import KMeans

inertia = []
k_range = range(1, 10)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=0)
    kmeans.fit(df)
    inertia.append(kmeans.inertia_)

fig = px.line(x=list(k_range), y=inertia, markers=True, title='Elbow Method')
fig.update_layout(xaxis_title='Number of clusters (k)', yaxis_title='Inertia')
fig.show()

In [26]:
kmeans = KMeans(n_clusters=3, random_state=0).fit(df)

In [27]:
df['cluster_label'] = kmeans.labels_
display(df.head())

,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex,cluster_label
0,39.1,18.7,181.0,3750.0,1,1
1,39.5,17.4,186.0,3800.0,0,1
2,40.3,18.0,195.0,3250.0,0,1
4,36.7,19.3,193.0,3450.0,0,1
5,39.3,20.6,190.0,3650.0,1,1


In [28]:
fig = px.scatter(df, x='flipper_length_mm', y='body_mass_g', color='cluster_label', title='Cluster Visualization')
fig.show()

In [50]:
# RODAR DBSCAN PROS PINGUINS
dbscan = DBSCAN()
df['dbscan_label'] = dbscan.fit_predict(df)

In [51]:
# RODAR GAUSSIAN MIXTURE PROS PINGUINS
gm = GaussianMixture(n_components=3, random_state=42)
df['gm_label'] = gm.fit_predict(df)

In [52]:
df.head()

,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex,cluster_label,dbscan_label,gm_label
0,39.1,18.7,181.0,3750.0,1,1,-1,0
1,39.5,17.4,186.0,3800.0,0,1,-1,0
2,40.3,18.0,195.0,3250.0,0,1,-1,0
4,36.7,19.3,193.0,3450.0,0,1,-1,0
5,39.3,20.6,190.0,3650.0,1,1,-1,0


In [54]:
ari_kmeans_dbscan = adjusted_rand_score(df['cluster_label'], df['dbscan_label'])
print(f"Adjusted Rand Index KMeans and DBSCAN: {ari_kmeans_dbscan}")

Adjusted Rand Index KMeans and DBSCAN: 0.0


In [55]:
ari_kmeans_gm = adjusted_rand_score(df['cluster_label'], df['gm_label'])
print(f"Adjusted Rand Index KMeans and Gaussian Mixture: {ari_kmeans_gm}")

Adjusted Rand Index KMeans and Gaussian Mixture: 0.9599557328803966


In [56]:
ari_dbscan_gm = adjusted_rand_score(df['dbscan_label'], df['gm_label'])
print(f"Adjusted Rand Index DBSCAN and Gaussian Mixture: {ari_dbscan_gm}")

Adjusted Rand Index DBSCAN and Gaussian Mixture: 0.0


# MOON Dataset

In [29]:
url_bd = 'https://raw.githubusercontent.com/reisanar/datasets/master/moons.csv'
moons = pd.read_csv(url_bd)

In [30]:
moons.head()

,X,Y
0,-0.415208,1.035735
1,0.058781,0.304334
2,1.109379,-0.509738
3,1.540948,-0.427550
4,0.929095,-0.532388


In [31]:
px.scatter(x=moons['X'], y=moons['Y'])

In [43]:
# RODAR KMEANS: https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html#sklearn.cluster.KMeans
km = kmeans = KMeans(n_clusters=4, random_state=42)

In [44]:
# RODAR DBSCAN: https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html
db = DBSCAN(eps=0.4)

In [45]:
# RODAR GAUSSIAN MIXTURE: https://scikit-learn.org/stable/modules/generated/sklearn.mixture.GaussianMixture.html#sklearn.mixture.GaussianMixture
gm = GaussianMixture(n_components=4, random_state=42)

In [46]:
km_c = km.fit_predict(moons)
gm_c = gm.fit_predict(moons)
db_c = db.fit_predict(moons)

In [47]:
# Plot original moons dataset
fig_moons = px.scatter(moons, x='X', y='Y', title='Original Moons Dataset')
fig_moons.show()

# KMeans clustering and plotting
fig_kmeans = px.scatter(moons, x='X', y='Y', color=km_c, title='KMeans Clustering')
fig_kmeans.show()

# DBSCAN clustering and plotting
fig_dbscan = px.scatter(moons, x='X', y='Y', color=db_c, title='DBSCAN Clustering')
fig_dbscan.show()

# Gaussian Mixture clustering and plotting
fig_gm = px.scatter(moons, x='X', y='Y', color=gm_c, title='Gaussian Mixture Clustering')
fig_gm.show()

In [49]:
df.head()

,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex,cluster_label
0,39.1,18.7,181.0,3750.0,1,1
1,39.5,17.4,186.0,3800.0,0,1
2,40.3,18.0,195.0,3250.0,0,1
4,36.7,19.3,193.0,3450.0,0,1
5,39.3,20.6,190.0,3650.0,1,1
